# v0.30.0 — Schema introspection

## 1. Intro

Since v0.30.0 the ORM reads the schema of a database as **immutable typed objects** — its tables with their fields,
indexes and events, and its accesses, analyzers, functions and params — and **generates `BaseSurrealModel` classes**
from it, as the full SurrealDB-ORM does:

- `await DatabaseIntrospector(connection).introspect()` returns a `SchemaState`;
- `ModelCodeGenerator().generate(schema)` turns it into the Python source of one model per table;
- `await generate_models_from_db(path, connection)` does both and writes the file.

It only reads: it sends `INFO FOR DB` and `INFO FOR TABLE`, never a statement that reads a record or changes the
schema. SurrealDB 2.x and 3.x render the same definition their own way (`option<int>` on 2.x, `none | int` on 3.x);
the reader normalises kinds, nullability, field paths and names, so the state and the generated types are the same on
both lines. The notebook runs top to bottom on both: only the server line (§2), the statements' texts (§5) and the
`COMPUTED` section (§8, SurrealDB 3.x only) differ.

## 2. Connect

`SURREALDB_HOST` and `SURREALDB_PORT` choose the server (default `localhost:8000`), over WebSocket. The schema lives in
its own database, `examples_introspection`, reached through the named connection `schema_demo`. The notebook starts
by resetting that database, so it can be re-executed in place: `REMOVE` and `DEFINE` go in **one** query, because on
SurrealDB 3.x a bare `REMOVE DATABASE` leaves the session failing until the database is defined again.

`apply_ddl()` runs SurrealQL through the SDK client's `query_raw()` and checks the status of **every** statement
(`client.query()` checks only the first one's).

In [1]:
import contextlib
import os

from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")
SERVER_URL = f"ws://{HOST}:{PORT}/rpc"

SurrealDBConnectionManager.set_connection(  # the default connection
    url=SERVER_URL,
    user="root",
    password="root",
    namespace="examples",
    database="examples",
)
SurrealDBConnectionManager.add_connection(  # the database this notebook reads
    "schema_demo",
    url=SERVER_URL,
    user="root",
    password="root",
    namespace="examples",
    database="examples_introspection",
)


async def apply_ddl(client, ddl: str) -> int:
    """Run SurrealQL statements and check every one of them; return how many ran."""
    response = await client.query_raw(ddl, {})
    if "error" in response:
        raise RuntimeError(response["error"])
    refused = [result["result"] for result in response["result"] if result["status"] != "OK"]
    if refused:
        raise RuntimeError(refused)
    return len(response["result"])


demo = await SurrealDBConnectionManager.get_client("schema_demo")
await apply_ddl(demo, "REMOVE DATABASE IF EXISTS examples_introspection; DEFINE DATABASE examples_introspection;")

version = str(await demo.version())  # "surrealdb-<major>.<minor>.<patch>"
line = int(version.removeprefix("surrealdb-").split(".")[0])
print("Connections:", SurrealDBConnectionManager.list_connections())
print(f"Server: SurrealDB {line}.x")

Connections: ['default', 'schema_demo']
Server: SurrealDB 3.x


## 3. A schema to read

A small shop schema, written once in SurrealQL and applied as is on both lines:

- `nb_company` and `nb_person` are `SCHEMAFULL` tables. `nb_person` uses most kinds the generator maps: optional
  kinds, a literal union, a `datetime` set by the server (`DEFAULT time::now() READONLY`, `VALUE time::now()`), an
  array, a nested object with a declared field (`address.city`), a `FLEXIBLE` object, a point, a duration, a decimal,
  a UUID, a record link, and two columns that are not Python names (`first-name`, `class`). It has a `UNIQUE` index
  and an event that writes to `nb_log`.
- `nb_likes` is a relation table (`TYPE RELATION IN nb_person OUT nb_person | nb_company`); `nb_log` is
  `SCHEMALESS`.
- `nb-weird` and `select` are valid SurrealDB tables the ORM cannot use: `table_name` refuses a name that is not an
  identifier, and the ORM's statements cannot name a table after a SurrealQL keyword. They are read all the same, and
  left out of the generated models (§6).
- The database also defines an analyzer, a function and a param.

In [2]:
SCHEMA = """
DEFINE TABLE nb_company SCHEMAFULL COMMENT 'Companies';
DEFINE FIELD name ON nb_company TYPE string;
DEFINE TABLE nb_person SCHEMAFULL;
DEFINE FIELD name ON nb_person TYPE string ASSERT string::len($value) > 0;
DEFINE FIELD email ON nb_person TYPE string VALUE string::lowercase($value);
DEFINE FIELD age ON nb_person TYPE option<int> DEFAULT 18;
DEFINE FIELD status ON nb_person TYPE 'active' | 'inactive' DEFAULT 'active';
DEFINE FIELD created ON nb_person TYPE datetime DEFAULT time::now() READONLY;
DEFINE FIELD updated ON nb_person TYPE datetime VALUE time::now();
DEFINE FIELD tags ON nb_person TYPE array<string>;
DEFINE FIELD address ON nb_person TYPE option<object>;
DEFINE FIELD address.city ON nb_person TYPE string;
DEFINE FIELD meta ON nb_person TYPE option<object> FLEXIBLE;
DEFINE FIELD location ON nb_person TYPE option<geometry<point>>;
DEFINE FIELD trial ON nb_person TYPE option<duration>;
DEFINE FIELD balance ON nb_person TYPE option<decimal>;
DEFINE FIELD token ON nb_person TYPE option<uuid>;
DEFINE FIELD employer ON nb_person TYPE option<record<nb_company>>;
DEFINE FIELD `first-name` ON nb_person TYPE option<string>;
DEFINE FIELD `class` ON nb_person TYPE option<string>;
DEFINE INDEX nb_person_email ON nb_person FIELDS email UNIQUE;
DEFINE EVENT nb_person_created ON nb_person WHEN $event = 'CREATE' THEN (CREATE nb_log SET msg = 'created');
DEFINE TABLE nb_likes TYPE RELATION IN nb_person OUT nb_person | nb_company SCHEMAFULL;
DEFINE FIELD since ON nb_likes TYPE option<datetime>;
DEFINE TABLE nb_log SCHEMALESS;
DEFINE TABLE `nb-weird` SCHEMALESS;
DEFINE TABLE `select` SCHEMALESS;
DEFINE ANALYZER nb_ascii TOKENIZERS class FILTERS lowercase, ascii;
DEFINE FUNCTION fn::nb_greet($name: string) { RETURN 'Hello ' + $name; };
DEFINE PARAM $nb_endpoint VALUE 'https://example.com';
"""
print("Statements applied:", await apply_ddl(demo, SCHEMA))

Statements applied: 30


## 4. Read the schema

`DatabaseIntrospector("schema_demo")` names the connection to read; `introspect()` sends `INFO FOR DB;`, then one
`INFO FOR TABLE` per table, outside any transaction. `schema.tables` holds one `TableState` per table, keyed and
iterated in codepoint order — `nb-weird` and `select` included.

In [3]:
from surreal_orm_lite import DatabaseIntrospector

schema = await DatabaseIntrospector("schema_demo").introspect()
print("connection:", schema.connection)
for name, table in schema.tables.items():
    print(f"  {name:<11} {table.schema_mode:<11} {table.table_type}")
likes = schema.tables["nb_likes"]
print("nb_likes relation:", likes.relation_in, "->", likes.relation_out)

connection: schema_demo
  nb-weird    SCHEMALESS  any
  nb_company  SCHEMAFULL  normal
  nb_likes    SCHEMAFULL  relation
  nb_log      SCHEMALESS  any
  nb_person   SCHEMAFULL  normal
  select      SCHEMALESS  any
nb_likes relation: ('nb_person',) -> ('nb_person', 'nb_company')


Each field is keyed by its path (`address.city`). `field_type` is its kind without the optional wrapper, `nullable`
says whether it may be `NONE`, and `default` is **SurrealQL text** (`'18'`, `"'active'"`, `'time::now()'`), as the
server renders it.

In [4]:
person = schema.tables["nb_person"]
print(f"{'field':<13} {'field_type':<22} {'nullable':<9} default")
for name, field in person.fields.items():
    print(f"{name:<13} {field.field_type:<22} {field.nullable!s:<9} {field.default}")

field         field_type             nullable  default
address       object                 True      None
address.city  string                 False     None
age           int                    True      18
balance       decimal                True      None
class         string                 True      None
created       datetime               False     time::now()
email         string                 False     None
employer      record<nb_company>     True      None
first-name    string                 True      None
location      geometry<point>        True      None
meta          object                 True      None
name          string                 False     None
status        'active' | 'inactive'  False     'active'
tags          array<string>          False     None
tags.*        string                 False     None
token         uuid                   True      None
trial         duration               True      None
updated       datetime               False     None


The table's index and event, and the database-level definitions: the function (keyed `fn::name`, with its parameter
names), the analyzer and the param (keyed without `$`). `schema.issues` lists what could not be read or parsed
— empty here. Printing a state shows names and structure only: definition texts, which may hold a secret (a param's
value), stay in their attributes, out of `repr()`.

In [5]:
index = person.indexes["nb_person_email"]
print("index:   ", index.name, "| fields", index.fields, "| unique", index.unique)
event = person.events["nb_person_created"]
print("event:   ", event.name, "| WHEN", event.when, "| THEN", event.then)
greet = schema.functions["fn::nb_greet"]
print("function:", greet.name, "| parameters", greet.parameters)
analyzer = schema.analyzers["nb_ascii"]
print("analyzer:", analyzer.name, "| tokenizers", analyzer.tokenizers, "| filters", analyzer.filters)
print("params:  ", list(schema.params))
print("issues:  ", schema.issues)
print()
print(schema.params["nb_endpoint"])

index:    nb_person_email | fields ('email',) | unique True
event:    nb_person_created | WHEN $event = 'CREATE' | THEN (CREATE nb_log SET msg = 'created')
function: fn::nb_greet | parameters ('name',)
analyzer: nb_ascii | tokenizers ('class',) | filters ('lowercase', 'ascii')
params:   ['nb_endpoint']
issues:   ()

ParamState(name='nb_endpoint', comment=None)


## 5. SurrealDB 2.x and 3.x

Every definition keeps the server's statement in `raw`. The two lines render the same `DEFINE FIELD` differently —
SurrealDB 2.x prints `TYPE option<int>`, 3.x `TYPE none | int` — and both give `field_type == "int"` and
`nullable is True`. Expressions (`DEFAULT`, `VALUE`, `ASSERT`, an event's `THEN` …) are kept **as each line renders
them**, not normalised; the generated annotations do not depend on them. The output below comes from the server the
notebook ran on.

In [6]:
age = person.fields["age"]
print("raw:       ", age.raw)
print("field_type:", age.field_type, "| nullable:", age.nullable, "| default:", age.default)
email = person.fields["email"]
print("raw:       ", email.raw)
print("value:     ", email.value)

raw:        DEFINE FIELD age ON nb_person TYPE none | int DEFAULT 18 PERMISSIONS FULL
field_type: int | nullable: True | default: 18
raw:        DEFINE FIELD email ON nb_person TYPE string VALUE string::lowercase($value) PERMISSIONS FULL
value:      string::lowercase($value)


## 6. Generate the models

`ModelCodeGenerator().generate(schema)` is pure: the same schema always gives the same source. Reading it:

- the header lists the tables left out, `nb-weird` and `select`, with the reason;
- one class per table — `NbLikes` is a `BaseSurrealEdge` (it declares no `id`, `in_` or `out`) whose `model_config`
  carries the table's ends, `relation_in=` and `relation_out=` (since v0.33.0), `NbLog` keeps undeclared fields
  (`extra="allow"`, a `SCHEMALESS` table), and every class carries `connection="schema_demo"`, since the schema was
  read from that connection; since v0.34.0 the table's clauses are keys too — `schema_mode=SchemaMode.SCHEMAFULL` on
  the `SCHEMAFULL` tables, `comment="Companies"` on `NbCompany`, `table_type=TableType.ANY` on `NbLog` — so
  `define_table()` can be called on the generated models;
- `address` becomes the sub-model `NbPersonAddress`; `first-name` and `class` become `first_name` and `class_` with
  an alias; `DEFAULT 18` and `DEFAULT 'active'` become Python defaults; `created` (`DEFAULT time::now()`) is listed in
  `server_fields`, so the server's default applies on create; `updated` (`VALUE time::now()`) is
  `Computed[datetime] = computed("time::now()")`, never written; `email`'s `VALUE` reads `$value`, so it stays a
  writable field the server transforms;
- `created` keeps a trailing `# READONLY` comment. `update()` keeps a top-level `READONLY` `datetime` the instance
  did not change: it writes back the stored nanoseconds, although Python reads microseconds. On SurrealDB 2.x,
  though, the server refuses for now a full replace (`update()`, `upsert()`) of a record with a typed `VALUE` field
  such as `updated` ([#267](https://github.com/EulogySnowfall/SurrealDB-ORM-lite/issues/267)): §7 changes `NbPerson`
  with `merge()`, which works on both lines.

In [7]:
from surreal_orm_lite import ModelCodeGenerator

source = ModelCodeGenerator().generate(schema)
print(source)

"""Models generated by surreal_orm_lite.generate_models_from_db() from a SurrealDB schema.

Review them before use. They describe tables that already exist: define_table()
keeps the clauses of the database they do not declare (a COMPUTED field stays
COMPUTED, a server DEFAULT stays), so it can be called on them; do not call
define_computed_fields() on them, it would replace the database's field definitions.
"""

from datetime import datetime
from decimal import Decimal
from typing import Any, Literal
from uuid import UUID

from pydantic import BaseModel, Field
from surreal_orm_lite import (
    BaseSurrealEdge,
    BaseSurrealModel,
    Computed,
    Duration,
    PointField,
    SchemaMode,
    SurrealConfigDict,
    TableType,
    computed,
)
from surrealdb import RecordID

# Not generated:
# - table 'nb-weird': its name is not an identifier, which table_name requires
# - table 'select': its name is a SurrealQL keyword the ORM cannot use as a table name


class NbCompany(BaseSurrealM

## 7. Write the file and use the models

`generate_models_from_db(path, "schema_demo")` reads the schema, generates the module and writes it. An existing file
is **replaced**, as the full ORM does, here atomically (a temporary file in the same directory, then `os.replace()`):
the second call below replaces the first one's file. `overwrite=False` refuses an existing path with
`FileExistsError`, before reading anything. The file lives in a temporary directory, whose path is hidden in the
output.

In [8]:
import pathlib
import tempfile

from surreal_orm_lite import generate_models_from_db

directory = tempfile.TemporaryDirectory()
path = pathlib.Path(directory.name) / "nb_models.py"
first = await generate_models_from_db(path, "schema_demo")
second = await generate_models_from_db(path, "schema_demo")  # replaces the file
print("written:", path.name, "| same source each time:", first == second == source == path.read_text())
try:
    await generate_models_from_db(path, "schema_demo", overwrite=False)
except FileExistsError as error:
    print(f"{type(error).__name__}: {str(error).replace(directory.name, '<tmp>')}")

written: nb_models.py | same source each time: True
FileExistsError: '<tmp>/nb_models.py' already exists: overwrite=False refuses to replace it.


Import the generated module and use its models against the database they describe: `NbPerson` routes to
`schema_demo` by itself. The server fills `created` and `updated`, which `save()` reads back, and stores `email`
lower-cased by its `VALUE string::lowercase($value)`, as a read shows.

In [9]:
import importlib.util
import sys

from surreal_orm_lite import GeometryPoint
from surrealdb import RecordID

spec = importlib.util.spec_from_file_location("nb_models", path)
nb_models = importlib.util.module_from_spec(spec)
sys.modules["nb_models"] = nb_models
spec.loader.exec_module(nb_models)
NbCompany, NbPerson, NbLog = nb_models.NbCompany, nb_models.NbPerson, nb_models.NbLog
print("NbPerson runs on:", repr(NbPerson.get_connection_name()))

await NbCompany(id="acme", name="Acme").save()
ada = await NbPerson(
    id="ada",
    name="Ada",
    email="Ada@Example.COM",
    tags=["math", "engines"],
    address=nb_models.NbPersonAddress(city="London"),
    location=GeometryPoint(-0.1276, 51.5072),
    employer=RecordID("nb_company", "acme"),
    first_name="Ada",
    class_="vip",
).save()
print("after save(): age", ada.age, "| status", ada.status, end="")
print(" | created", type(ada.created).__name__, "| updated", type(ada.updated).__name__)

stored = await NbPerson.objects().get("ada")
print("read back:  ", stored.email, "|", stored.address.city, "|", stored.employer, "|", stored.first_name, stored.class_)
print("by alias: ", {key: stored.model_dump(by_alias=True)[key] for key in ("first-name", "class")})

NbPerson runs on: 'schema_demo'
after save(): age 18 | status active | created datetime | updated datetime
read back:   ada@example.com | London | nb_company:acme | Ada vip
by alias:  {'first-name': 'Ada', 'class': 'vip'}


`merge()` changes only the fields it names (`UPDATE … MERGE`), on both lines: the server keeps `created` as stored and
recomputes `updated`. The event wrote one `nb_log` row when Ada was created: `NbLog` declares no field, so the row's
`msg` is in `model_extra`.

In [10]:
await ada.merge(name="Ada Lovelace", age=36)
again = await NbPerson.objects().get("ada")
print("after merge():", again.name, again.age, "| created unchanged:", again.created == ada.created)

logs = await NbLog.objects().all()
print("nb_log rows:", [log.model_extra for log in logs])

after merge(): Ada Lovelace 36 | created unchanged: True
nb_log rows: [{'msg': 'created'}]


`connection=` is the **name** of a connection — the one given to `add_connection()` — never a client object: a
client does not carry the ORM's connection name, which routing, reconnects and the generated `connection=` key need.
Passing one raises `TypeError` with the fix.

In [11]:
try:
    DatabaseIntrospector(demo)
except TypeError as error:
    print(f"{type(error).__name__}: {error}")

TypeError: connection= takes the name of a registered connection, not a client object: pass the name it was added with (add_connection()), or call DatabaseIntrospector() inside SurrealDBConnectionManager.using(name).


## 8. `COMPUTED` fields (SurrealDB 3.x only)

SurrealDB 3.x has a `COMPUTED` clause: the value is computed on every read and never stored. The reader puts its
expression in `FieldState.computed` and the generator writes `Computed[T] = computed("…")`. SurrealDB 2.x refuses the
clause, so the cell below probes it with a scratch `DEFINE FIELD`: on 3.x the field is read and generated, on 2.x the
section is skipped (a 3.x server that refuses it is an error).

In [12]:
await apply_ddl(
    demo,
    "DEFINE TABLE nb_invoice SCHEMAFULL;"
    "DEFINE FIELD price ON nb_invoice TYPE float;"
    "DEFINE FIELD qty ON nb_invoice TYPE int;",
)
try:
    await apply_ddl(demo, "DEFINE FIELD total ON nb_invoice TYPE float COMPUTED price * qty;")
    has_computed = True
except Exception:
    has_computed = False
if line >= 3 and not has_computed:
    raise RuntimeError("SurrealDB 3.x refused COMPUTED: fix the statement")

if has_computed:
    invoices = await DatabaseIntrospector("schema_demo", tables=["nb_invoice"]).introspect()
    total = invoices.tables["nb_invoice"].fields["total"]
    print("total:", "field_type", total.field_type, "| computed", repr(total.computed), "| value", total.value)
    print()
    print(ModelCodeGenerator().generate(invoices))
else:
    print(f"SurrealDB {line}.x has no COMPUTED clause: skipped.")

total: field_type float | computed 'price * qty' | value None

"""Models generated by surreal_orm_lite.generate_models_from_db() from a SurrealDB schema.

Review them before use. They describe tables that already exist: define_table()
keeps the clauses of the database they do not declare (a COMPUTED field stays
COMPUTED, a server DEFAULT stays), so it can be called on them; do not call
define_computed_fields() on them, it would replace the database's field definitions.
"""

from surreal_orm_lite import (
    BaseSurrealModel,
    Computed,
    SchemaMode,
    SurrealConfigDict,
    computed,
)


class NbInvoice(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="nb_invoice", schema_mode=SchemaMode.SCHEMAFULL, connection="schema_demo")

    id: str | None = None
    price: float
    qty: int
    total: Computed[float] = computed("price * qty")



## 9. Cleanup

Remove the temporary file and the `examples_introspection` database, unregister `schema_demo` —
`remove_connection()` closes its client — then close every connection with `close_all_connections()`.

In [13]:
directory.cleanup()
sys.modules.pop("nb_models", None)
await apply_ddl(demo, "REMOVE DATABASE IF EXISTS examples_introspection;")
await SurrealDBConnectionManager.remove_connection("schema_demo")
print("Connections:", SurrealDBConnectionManager.list_connections())
await SurrealDBConnectionManager.close_all_connections()
print("Done")

Connections: ['default']
Done
